# 长度计算为什么关心交叉项？

展开两份变化相加的平方长度，就出现内积；正交使交叉项消失。先比较位置误差与单位基坐标，再实际计算同一矩阵的四空间。输入零方向与输出左零方向虽都写三项，含义不同；后者决定目标是否相容。

从新内核按顺序运行；没有滑块时可直接修改参数。计算单元用于核对前面的推理，参考结果放在相应尝试之后。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def check_prediction(value, expected, label):
    if value is None:
        print(label + '：尚未填写，请先在纸上计算，再替换 None。')
        return None
    try:
        proposed = np.asarray(value, dtype=float)
    except (TypeError, ValueError, OverflowError):
        print(label + '：请填写数值、数值列表或数组。')
        return False
    reference = np.asarray(expected, dtype=float)
    if proposed.shape != reference.shape:
        print(label + '：尺寸应为 ' + str(reference.shape) + '，请检查行列。')
        return False
    if not np.all(np.isfinite(proposed)):
        print(label + '：请检查是否除以零或产生了非有限数。')
        return False
    correct = bool(np.allclose(proposed, reference))
    print(label + ('：核对正确。' if correct else '：尚不一致，请回到方程或内积检查。'))
    return correct


## 两个位置误差

(2,4)、(3,2) 到 (3,4) 的距离各多少像素？


In [ ]:
target = np.array([3.,4.])
for estimate in [np.array([2.,4.]),np.array([3.,2.])]:
    error = target-estimate
    print('误差:', error, '距离:', np.linalg.norm(error))


## 正交基的长度

先算 p、q 的内积与长度，再归一化。


In [ ]:
p,q = np.array([1.,1.]),np.array([1.,-1.])
print('内积:', p @ q, '长度:', np.linalg.norm(p),np.linalg.norm(q))
Q = np.column_stack((p/np.linalg.norm(p),q/np.linalg.norm(q)))
assert np.allclose(Q.T @ Q,np.eye(2))
x = np.array([3.,1.])
coordinates = Q.T @ x
print('单位基坐标:',coordinates)
assert np.allclose(Q @ coordinates,x)
assert np.allclose(np.linalg.norm(coordinates),np.linalg.norm(x))


## 单位改变时，距离不能盲目沿用

面积差 10 平方米、距离差 1 公里。换为米后，若尺度也换为 1000 米，标准化误差是否一致？


In [ ]:
raw_km = np.array([10.,1.])
raw_m = np.array([10.,1000.])
print('盲目计算的两个长度:',np.linalg.norm(raw_km),np.linalg.norm(raw_m))
scaled_km = raw_km / np.array([10.,1.])
scaled_m = raw_m / np.array([10.,1000.])
assert np.allclose(scaled_km,scaled_m)
print('指定尺度后的长度:',np.linalg.norm(scaled_km))


## 四空间必须在两侧分别检查

C 记录前两项之和、后两项之和以及它们的和。第三条测量重复。先在纸上求行空间、零空间、列空间、左零空间的基。

输入与输出都写成三项，不代表两侧的三个坐标含义一样。


In [ ]:
C4 = np.array([[1.,1.,0.],[0.,1.,1.],[1.,2.,1.]])
row_basis = np.array([[1.,1.,0.],[0.,1.,1.]]).T
column_basis = C4[:, [0,2]]
input_null = np.array([1.,-1.,1.])
output_null = np.array([-1.,-1.,1.])
assert np.allclose(C4 @ input_null, 0)
assert np.allclose(C4.T @ output_null, 0)
assert np.allclose(row_basis.T @ input_null, 0)
assert np.allclose(column_basis.T @ output_null, 0)
assert np.linalg.matrix_rank(row_basis) == 2
assert np.linalg.matrix_rank(column_basis) == 2
print('input-side basis:\n', row_basis, 'invisible input:', input_null)
print('output-side basis:\n', column_basis, 'output relation:', output_null)


## 先预测哪个目标相容

目标 (2,3,5) 与 (2,3,6) 中，哪个与整个左零空间正交？把两次与左零基向量的内积按顺序填入 `predicted_relations`。

反例检查：如果误用输入零方向做相容检查，会不会得到相同判断？


In [ ]:
predicted_relations = None  # List of two scalar inner products.
check_prediction(predicted_relations, [0,1], '相容关系预测')
for target in [np.array([2.,3.,5.]), np.array([2.,3.,6.])]:
    print('target:', target, 'left-null test:', output_null @ target,
          'wrong input-null test:', input_null @ target)
assert np.isclose(output_null @ np.array([2.,3.,5.]), 0)
assert not np.isclose(output_null @ np.array([2.,3.,6.]), 0)
assert np.allclose(C4 @ np.array([2.,0.,3.]), [2,3,5])


<details><summary>核对与重建</summary>
两次相容关系为 0、1。零方向是输入变化；左零方向是输出记录之间的关系。把两侧换用，会把可达目标误判。再解释单位基下 (3,1) 的坐标为什么是 (2√2,√2)，而不是非单位基下的 (2,1)。
</details>


## 改变条件以后，理由还成立吗？

1. 同一个向量在非单位基和单位基下的坐标为什么不同？
2. 对C，分别从方程求出输入零方向与输出左零方向，并证明它们各与哪组基正交。
3. 指出一个不可达目标，不调用求解器，只用输出相容关系解释。
4. 哪些结论只需内积与消元，哪些一般存在性结论还需要谱定理？
